# Round 9: Catboost

In [1]:
## ADD SAVE PARAMETERS
save = True
save_name = "round9_CatBoost"
notes = "CatBoost. optuna 200 rounds. Linear FE features"

In [2]:
# import statements
import numpy as np
import pandas as pd
import json, os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import optuna


In [3]:
# load processed data
X_train = pd.read_csv("../data/processed/feature_engineering/X_train_fe_linear.csv")
X_test = pd.read_csv("../data/processed/feature_engineering/X_test_fe_linear.csv")
Y_train = pd.read_csv("../data/processed/Y_train.csv")

In [4]:
# train model - no optuna

import catboost as cb
model = cb.CatBoostRegressor(
    iterations    = 2000,
    learning_rate = 0.02,
    depth         = 6,
    random_seed   = 42,
    verbose       = 0
)
model.fit(X_train, Y_train)

1.26.4


CatBoostRegressor(depth=6, iterations=2000, learning_rate=0.02, loss_function='RMSE', random_seed=42, verbose=0)

In [5]:
# save to output

preds = np.expm1(model.predict(X_test))

submission = pd.DataFrame({
    'Id'       : pd.read_csv('../data/raw/test.csv')['Id'],
    'SalePrice': preds
})

submission.to_csv(f'../data/output/{save_name}.csv', index=False)
print(submission.head())

     Id      SalePrice
0  1461  127415.285181
1  1462  165734.259328
2  1463  178584.547584
3  1464  195360.028025
4  1465  180599.767319


In [6]:
# get cv_scores

cv_scores = cross_val_score(
    model, X_train, Y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

print(f"CV RMSE scores : {-cv_scores}")
print(f"Mean CV RMSE   : {-cv_scores.mean():.4f}")
print(f"Std CV RMSE    : {cv_scores.std():.4f}")

CV RMSE scores : [0.10548397 0.12871803 0.12662904 0.1081435  0.12806477]
Mean CV RMSE   : 0.1194
Std CV RMSE    : 0.0103


In [7]:
# save

log_entry = {
    "model"       : save_name,
    "cv_rmse_mean": round(float(-cv_scores.mean()), 5),
    "cv_rmse_std" : round(float(cv_scores.std()), 5),
    "cv_scores"   : [round(float(-s), 5) for s in cv_scores],
    "params"      : {},          # if using optuna, else {}
    "submission"  : f"{save_name}.csv",
    "notes"       : notes
}

if save: 
    os.makedirs('../data/output', exist_ok=True)
    log_path = '../data/output/cv_scores.json'
    # Load existing log or start fresh
    if os.path.exists(log_path):
        with open(log_path, 'r') as f:
            log = json.load(f)
    else:
        log = []

    log.append(log_entry)

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)

    print(f"Logged CV score: {log_entry['cv_rmse_mean']:.5f} ± {log_entry['cv_rmse_std']:.5f}")
else: 
    print(f"Not logged, change to save boolean to log: {log_entry['cv_rmse_mean']:.5f} ± {log_entry['cv_rmse_std']:.5f}")

Logged CV score: 0.11941 ± 0.01034


In [8]:
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

kf  = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(X_train))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr,  X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    Y_tr,  Y_val = Y_train.iloc[tr_idx], Y_train.iloc[val_idx]

    model.fit(X_tr, Y_tr)
    oof[val_idx] = model.predict(X_val)

    fold_rmse = root_mean_squared_error(Y_val, oof[val_idx])
    print(f"  Fold {fold+1} RMSE: {fold_rmse:.5f}")

oof_rmse = root_mean_squared_error(Y_train, oof)
print(f"\n✅ OOF RMSE : {oof_rmse:.5f}")

# ── Save ─────────────────────────────────────────────────────
if save: 
    np.save(f'../data/output/oof/oof_{save_name}.npy', oof)
    print(f"✅ OOF saved to ../data/output/oof/oof_{save_name}.npy")

  Fold 1 RMSE: 0.12881
  Fold 2 RMSE: 0.11207
  Fold 3 RMSE: 0.15562
  Fold 4 RMSE: 0.11691
  Fold 5 RMSE: 0.10707

✅ OOF RMSE : 0.12530
✅ OOF saved to ../data/output/oof/oof_round9_CatBoost.npy
